# DeepTrace - Synthetic Enterprise Log Generator

## Objective

This notebook generates a realistic synthetic enterprise cybersecurity dataset for the DeepTrace Behavioral Threat Detection System.

The generated dataset simulates:

- Normal employee behavior
- Department-specific activity patterns
- Role-based permissions
- User behavioral drift
- Benign variations
- Insider threat scenarios
- Credential theft attacks
- Privilege escalation
- Data exfiltration
- Brute-force attacks
- Impossible travel
- Malware execution
- Enterprise logging noise

The generated dataset will be used for:

- Isolation Forest (Anomaly Detection)
- Transformer-based Behavioral Modeling
- XGBoost Threat Classification
- SHAP Explainability
- Dashboard Analytics

---

## Dataset Goals

- Realistic enterprise behavior
- Configurable organization size
- Multiple departments and roles
- Event-level logging
- Controlled attack injection
- Controlled noise injection
- Scalable to hundreds of thousands of events

In [2]:
!pip install faker


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 62.3 MB/s eta 0:00:00


In [3]:
import random
import uuid
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

from faker import Faker
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

fake = Faker()
Faker.seed(42)

# Configuration

This section defines the global parameters used throughout the synthetic data generation process.

Changing these values allows the simulator to generate organizations of different sizes without modifying the generation logic.

In [4]:
CONFIG = {

    # Organization
    "company_name": "DeepTrace Corporation",

    # Simulation
    "num_employees": 500,
    "simulation_days": 30,

    # Working Days
    "working_days": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"],

    # Average Events
    "min_events_per_day": 15,
    "max_events_per_day": 45,

    # Attack Distribution
    "attack_percentage": 0.10,

    # Noise
    "missing_value_percentage": 0.02,
    "duplicate_event_percentage": 0.01,
    "timestamp_jitter_seconds": 120,

    # Random Seed
    "seed": 42
}

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

# Organization Structure

This section defines the enterprise environment that will be simulated.

Every employee generated later will belong to one department, one role, one office location, and will use a department-specific set of applications and resources.

This allows the generated activity logs to resemble realistic enterprise behavior instead of completely random events.

In [5]:
DEPARTMENTS = {
    "Engineering": {
        "roles": ["Software Engineer", "Senior Engineer", "Engineering Manager"],
        "applications": ["GitHub", "Jira", "VS Code", "Docker", "Confluence"],
        "resources": ["Source Code", "CI/CD Pipeline", "Cloud Infrastructure"],
        "devices": ["Windows Laptop", "MacBook", "Linux Workstation"],
        "working_hours": (10, 19)
    },

    "Finance": {
        "roles": ["Financial Analyst", "Accountant", "Finance Manager"],
        "applications": ["SAP", "Excel", "Power BI", "Oracle DB"],
        "resources": ["Financial Reports", "Invoices", "Payroll"],
        "devices": ["Windows Laptop", "Desktop"],
        "working_hours": (9, 18)
    },

    "HR": {
        "roles": ["HR Executive", "HR Manager"],
        "applications": ["HRMS", "Outlook", "SharePoint"],
        "resources": ["Employee Records", "Payroll", "Policies"],
        "devices": ["Windows Laptop"],
        "working_hours": (9, 18)
    },

    "IT": {
        "roles": ["System Administrator", "Network Engineer", "IT Manager"],
        "applications": ["Active Directory", "VPN", "PowerShell", "Server Console"],
        "resources": ["Servers", "Network Devices", "Security Logs"],
        "devices": ["Windows Laptop", "Linux Workstation"],
        "working_hours": (8, 18)
    },

    "Security": {
        "roles": ["SOC Analyst", "Security Engineer", "Incident Responder"],
        "applications": ["SIEM", "EDR", "Firewall Console", "Threat Intelligence"],
        "resources": ["Security Alerts", "Threat Reports", "Incident Database"],
        "devices": ["Windows Laptop", "Linux Workstation"],
        "working_hours": (8, 17)
    },

    "Sales": {
        "roles": ["Sales Executive", "Sales Manager"],
        "applications": ["CRM", "Outlook", "Teams"],
        "resources": ["Customer Database", "Sales Reports"],
        "devices": ["Windows Laptop", "Mobile"],
        "working_hours": (9, 18)
    },

    "Legal": {
        "roles": ["Legal Advisor", "Legal Manager"],
        "applications": ["Document Management", "Outlook"],
        "resources": ["Legal Documents", "Contracts"],
        "devices": ["Windows Laptop"],
        "working_hours": (9, 18)
    },

    "Executive": {
        "roles": ["Director", "Vice President", "CEO"],
        "applications": ["Outlook", "Power BI", "Teams"],
        "resources": ["Executive Reports", "Business Dashboard"],
        "devices": ["MacBook", "Windows Laptop"],
        "working_hours": (9, 18)
    }
}

In [6]:
OFFICE_LOCATIONS = [
    "Bangalore",
    "Mumbai",
    "Delhi",
    "Hyderabad",
    "Pune",
    "Chennai"
]

EVENT_TYPES = [
    "Login",
    "Logout",
    "Email",
    "File Access",
    "Database Query",
    "Download",
    "Upload",
    "VPN Login",
    "USB Insert",
    "Admin Action",
    "PowerShell Execution",
    "Password Change",
    "Print",
    "Application Launch"
]

NETWORK_TYPES = [
    "Corporate LAN",
    "VPN",
    "Home WiFi",
    "Public WiFi"
]

AUTH_METHODS = [
    "Password",
    "MFA",
    "SSO",
    "Biometric"
]

LABELS = [
    "Normal",
    "Threat"
]

In [7]:
ATTACK_TYPES = {
    "Credential Theft": 0.20,
    "Brute Force": 0.15,
    "Privilege Escalation": 0.15,
    "Impossible Travel": 0.10,
    "Insider Threat": 0.15,
    "Data Exfiltration": 0.15,
    "Malware Execution": 0.05,
    "Lateral Movement": 0.05
}

# Employee Generation

This section creates the synthetic workforce for the simulated enterprise.

Each employee is assigned:

- Unique Employee ID
- Full Name
- Department
- Role
- Office Location
- Preferred Device
- Working Hours
- Primary Applications
- Primary Resources
- Risk Baseline
- Typical Login Time
- Typical Logout Time

The generated employee table acts as the master reference for all subsequent event generation.

In [31]:
def generate_employee_id(index):
    return f"EMP{index:04d}"


def random_login_time(start_hour):
    hour = random.randint(start_hour - 1, start_hour + 1)
    minute = random.randint(0, 59)
    return f"{hour:02d}:{minute:02d}"


def random_logout_time(end_hour):
    hour = random.randint(end_hour - 1, end_hour + 1)
    minute = random.randint(0, 59)
    return f"{hour:02d}:{minute:02d}"


def assign_risk_baseline(role):

    high_risk = [
    "CEO",
    "CTO",
    "CFO",
    "Director",
    "Administrator",
    "Security Engineer",
    "Security Analyst",
    "Network Engineer"
]

    medium_risk = [
        "Manager",
        "Team Lead",
        "HR Manager",
        "Finance Manager"
    ]

    if any(x in role for x in high_risk):
        return "High"

    if any(x in role for x in medium_risk):
        return "Medium"

    return "Low"

def assign_access_level(role):

    if any(x in role for x in ["CEO", "Director", "Administrator"]):
        return "Admin"

    if "Manager" in role:
        return "Elevated"

    return "Standard"

In [37]:
employees = []

for i in range(1, CONFIG["num_employees"] + 1):

    department = random.choice(list(DEPARTMENTS.keys()))
    dept_info = DEPARTMENTS[department]

    if department == "Executive":
        role = random.choices(
            dept_info["roles"],
            weights=[15,4,1],
            k=1
        )[0]
    else:
        role = random.choice(dept_info["roles"])

    start_hour, end_hour = dept_info["working_hours"]

    employee = {

        "employee_id": generate_employee_id(i),

        "name": fake.name(),

        "department": department,

        "role": role,

        "office_location": random.choice(OFFICE_LOCATIONS),

        "preferred_device": random.choice(dept_info["devices"]),

        "working_start": start_hour,

        "working_end": end_hour,

        "typical_login": random_login_time(start_hour),

        "typical_logout": random_logout_time(end_hour),

        "applications": dept_info["applications"],

        "resources": dept_info["resources"],

        "risk_baseline": assign_risk_baseline(role),

        "access_level": assign_access_level(role)

    }

    employees.append(employee)

employees_df = pd.DataFrame(employees)

In [38]:
print(f"Total Employees : {len(employees_df)}")

print("\nDepartments")
display(employees_df["department"].value_counts())

print("\nRoles")
display(employees_df["role"].value_counts())

print("\nSample Employees")
display(employees_df.head())

Total Employees : 500

Departments


,count
department,
Engineering,80
Security,68
Legal,66
Finance,66
Executive,65
HR,55
IT,52
Sales,48



Roles


,count
role,
Director,50
Legal Manager,43
Software Engineer,30
HR Executive,29
Incident Responder,28
Sales Executive,28
Finance Manager,26
HR Manager,26
Engineering Manager,25



Sample Employees


,employee_id,name,department,role,office_location,preferred_device,working_start,working_end,typical_login,typical_logout,applications,resources,risk_baseline,access_level
0,EMP0001,Steven Medina,Security,Incident Responder,Mumbai,Linux Workstation,8,17,08:01,16:08,"[SIEM, EDR, Firewall Console, Threat Intellige...","[Security Alerts, Threat Reports, Incident Dat...",Low,Standard
1,EMP0002,Annette Lewis,Finance,Finance Manager,Delhi,Desktop,9,18,09:52,19:53,"[SAP, Excel, Power BI, Oracle DB]","[Financial Reports, Invoices, Payroll]",Medium,Elevated
2,EMP0003,Michelle Richardson,IT,IT Manager,Chennai,Windows Laptop,8,18,07:54,18:13,"[Active Directory, VPN, PowerShell, Server Con...","[Servers, Network Devices, Security Logs]",Medium,Elevated
3,EMP0004,Jessica Rosario,Legal,Legal Manager,Hyderabad,Windows Laptop,9,18,09:08,18:22,"[Document Management, Outlook]","[Legal Documents, Contracts]",Medium,Elevated
4,EMP0005,Henry Gray,Sales,Sales Executive,Hyderabad,Mobile,9,18,10:24,19:15,"[CRM, Outlook, Teams]","[Customer Database, Sales Reports]",Low,Standard


# Behavioral Profiles

This section defines how employees behave based on their department.

Instead of generating completely random events, each department follows a realistic behavioral profile including:

- Frequently used applications
- Common event types
- Typical network usage
- Working-hour preferences
- Expected resource access patterns

These behavioral profiles serve as the baseline for generating realistic enterprise activity logs.

In [11]:
BEHAVIOR_PROFILES = {

    "Engineering": {
        "common_events": {
            "Application Launch": 0.25,
            "File Access": 0.20,
            "Download": 0.15,
            "Upload": 0.15,
            "Login": 0.10,
            "Logout": 0.10,
            "Database Query": 0.05
        },
        "preferred_network": "Corporate LAN"
    },

    "Finance": {
        "common_events": {
            "Application Launch": 0.20,
            "Database Query": 0.20,
            "File Access": 0.20,
            "Email": 0.15,
            "Print": 0.10,
            "Login": 0.10,
            "Logout": 0.05
        },
        "preferred_network": "Corporate LAN"
    },

    "HR": {
        "common_events": {
            "Email": 0.30,
            "File Access": 0.25,
            "Application Launch": 0.15,
            "Print": 0.10,
            "Login": 0.10,
            "Logout": 0.10
        },
        "preferred_network": "Corporate LAN"
    },

    "IT": {
        "common_events": {
            "Admin Action": 0.20,
            "PowerShell Execution": 0.20,
            "VPN Login": 0.15,
            "Application Launch": 0.15,
            "File Access": 0.10,
            "Login": 0.10,
            "Logout": 0.10
        },
        "preferred_network": "Corporate LAN"
    },

    "Security": {
        "common_events": {
            "Application Launch": 0.20,
            "File Access": 0.15,
            "Database Query": 0.15,
            "VPN Login": 0.15,
            "Admin Action": 0.15,
            "Login": 0.10,
            "Logout": 0.10
        },
        "preferred_network": "Corporate LAN"
    },

    "Sales": {
        "common_events": {
            "Email": 0.30,
            "Application Launch": 0.20,
            "File Access": 0.15,
            "Upload": 0.15,
            "Login": 0.10,
            "Logout": 0.10
        },
        "preferred_network": "Home WiFi"
    },

    "Legal": {
        "common_events": {
            "File Access": 0.30,
            "Email": 0.20,
            "Print": 0.20,
            "Application Launch": 0.20,
            "Login": 0.05,
            "Logout": 0.05
        },
        "preferred_network": "Corporate LAN"
    },

    "Executive": {
        "common_events": {
            "Email": 0.25,
            "Application Launch": 0.25,
            "Database Query": 0.15,
            "File Access": 0.15,
            "VPN Login": 0.10,
            "Login": 0.05,
            "Logout": 0.05
        },
        "preferred_network": "Corporate LAN"
    }
}

# Enterprise Event Generation

This section generates realistic daily enterprise activity for every employee.

For each simulated working day, employees perform activities based on their behavioral profile.

Generated events include:

- Login / Logout
- Application Usage
- File Access
- Email Activity
- Database Queries
- Downloads / Uploads
- Administrative Actions
- VPN Access

Each event is timestamped and linked to the corresponding employee, department, role, application, resource, and network.

In [41]:
def choose_event(department):

    events = list(BEHAVIOR_PROFILES[department]["common_events"].keys())

    probabilities = list(BEHAVIOR_PROFILES[department]["common_events"].values())

    return np.random.choice(events, p=probabilities)

In [44]:
# Generate realistic daily enterprise session
def generate_daily_events(employee, current_date):

    events = []

    session_id = str(uuid.uuid4())

    start_hour = employee["working_start"]
    end_hour = employee["working_end"]

    login_time = datetime(
        current_date.year,
        current_date.month,
        current_date.day,
        random.randint(start_hour - 1, start_hour),
        random.randint(0, 59),
        random.randint(0, 59)
    )

    logout_time = datetime(
        current_date.year,
        current_date.month,
        current_date.day,
        random.randint(end_hour, end_hour + 1),
        random.randint(0, 59),
        random.randint(0, 59)
    )

    num_middle_events = random.randint(12, 35)

    session_events = ["Login"]

    for _ in range(num_middle_events):

        event = choose_event(employee["department"])

        if event not in ["Login", "Logout"]:
            session_events.append(event)

    session_events.append("Logout")

    total_seconds = int((logout_time - login_time).total_seconds())

    step = max(60, total_seconds // len(session_events))

    current_timestamp = login_time

    for event_type in session_events:

        if event_type == "Application Launch":
            application = random.choice(employee["applications"])
        else:
            application = random.choice(
                EVENT_APPLICATION_MAPPING.get(
                    event_type,
                    employee["applications"]
                )
            )

        resource = random.choice(
            EVENT_RESOURCE_MAPPING.get(
                event_type,
                employee["resources"]
            )
        )

        if event_type == "VPN Login":
            network = "VPN"
            auth = "MFA"

        elif event_type == "Admin Action":
            network = "Corporate LAN"
            auth = "MFA"

        elif event_type == "Login":
            network = BEHAVIOR_PROFILES[
                employee["department"]
            ]["preferred_network"]
            auth = "SSO"

        else:
            network = BEHAVIOR_PROFILES[
                employee["department"]
            ]["preferred_network"]

            if employee["access_level"] == "Admin":
                auth = "MFA"

            elif employee["access_level"] == "Elevated":
                auth = random.choice(["SSO", "MFA"])

            else:
                auth = random.choice(
                    ["Password", "SSO", "Biometric"]
                )

        events.append({

            "event_id": str(uuid.uuid4()),

            "session_id": session_id,

            "timestamp": current_timestamp,

            "employee_id": employee["employee_id"],

            "employee_name": employee["name"],

            "department": employee["department"],

            "role": employee["role"],

            "office_location": employee["office_location"],

            "device": employee["preferred_device"],

            "network_type": network,

            "authentication": auth,

            "event_type": event_type,

            "application": application,

            "resource": resource,

            "risk_baseline": employee["risk_baseline"],

            "access_level": employee["access_level"],

            "label": "Normal"

        })

        current_timestamp += timedelta(
            seconds=step + random.randint(-300, 600)
        )

    events.sort(key=lambda x: x["timestamp"])

    events[-1]["event_type"] = "Logout"

    return events

# Generate Enterprise Activity

This section simulates the complete organization over the configured time period.

For every working day:

- Every employee performs daily activities.
- Events are generated according to department-specific behavioral profiles.
- Each event is stored as a structured enterprise security log.

The resulting dataset forms the baseline ("normal") enterprise behavior before cyber attacks and noise are injected.

In [45]:
all_events = []

start_date = datetime.today() - timedelta(days=CONFIG["simulation_days"])

for day in tqdm(range(CONFIG["simulation_days"])):

    current_date = start_date + timedelta(days=day)

    weekday = current_date.strftime("%A")

    # Skip weekends
    if weekday not in CONFIG["working_days"]:
        continue

    for _, employee in employees_df.iterrows():

        daily_events = generate_daily_events(employee, current_date)

        all_events.extend(daily_events)

events_df = pd.DataFrame(all_events)

events_df = events_df.sort_values("timestamp").reset_index(drop=True)

100%|██████████| 30/30 [00:19<00:00,  1.56it/s]


In [46]:
print(f"Total Events Generated : {len(events_df):,}")

print("\nSample Events")

display(events_df.head())

print("\nEvent Distribution")

display(events_df["event_type"].value_counts())

print("\nDepartment Distribution")

display(events_df["department"].value_counts())

Total Events Generated : 238,606

Sample Events


,event_id,session_id,timestamp,employee_id,employee_name,department,role,office_location,device,network_type,authentication,event_type,application,resource,risk_baseline,access_level,label
0,72a9289d-ee16-48dd-8b3d-9fb63372b154,19daedc5-bb24-441f-93a3-7a9dc5539d4a,2026-06-25 07:00:24,EMP0471,Heather Miller,Security,SOC Analyst,Chennai,Linux Workstation,Corporate LAN,SSO,Login,Authentication Service,Authentication Server,Low,Standard,Normal
1,68104a3b-b23b-4f12-942b-2be1bedd4d4d,78341bbb-0430-476f-aaaa-f31289f87df4,2026-06-25 07:00:25,EMP0401,David Peterson,IT,IT Manager,Delhi,Windows Laptop,Corporate LAN,SSO,Login,Authentication Service,Authentication Server,Medium,Elevated,Normal
2,43430bee-646c-42d9-97a4-7305a956606e,9b8f0207-0b81-4f75-96c0-1d06887678b5,2026-06-25 07:00:33,EMP0029,Marco Davis,IT,IT Manager,Mumbai,Windows Laptop,Corporate LAN,SSO,Login,Authentication Service,Authentication Server,Medium,Elevated,Normal
3,a3df2883-010d-4a12-ac11-bf8534e69025,78878d50-ac38-4ada-8135-45a67be92cdc,2026-06-25 07:00:37,EMP0277,Melody Griffin,Security,Security Engineer,Pune,Linux Workstation,Corporate LAN,SSO,Login,Authentication Service,Authentication Server,High,Standard,Normal
4,c397360b-7c39-498d-91af-25001a7ce56a,51b68bc6-3a50-4ba4-a989-3ffd10754b73,2026-06-25 07:01:20,EMP0015,Brian Kim DDS,IT,System Administrator,Hyderabad,Windows Laptop,Corporate LAN,SSO,Login,Authentication Service,Authentication Server,High,Admin,Normal



Event Distribution


,count
event_type,
Application Launch,52762
File Access,49092
Email,36821
Database Query,19626
Print,13362
VPN Login,12875
Logout,11000
Login,11000
Admin Action,10495



Department Distribution


,count
department,
Engineering,36522
Legal,33930
Executive,33627
Finance,32444
Security,31285
HR,25080
IT,23842
Sales,21876


In [101]:
EVENT_APPLICATION_MAPPING = {
    "Login": ["Authentication Service"],
    "Login Failed": ["Authentication Service"],
    "Logout": ["Authentication Service"],
    "Email": ["Outlook"],
    "File Access": ["File Explorer", "SharePoint"],
    "Database Query": ["Oracle DB", "SQL Server", "SAP"],
    "Download": ["Browser", "SharePoint"],
    "Upload": ["SharePoint", "OneDrive"],
    "VPN Login": ["VPN Client"],
    "Admin Action": ["Active Directory", "Server Console"],
    "PowerShell Execution": ["PowerShell"],
    "Print": ["Print Service"],
    "Application Launch": [
        "CRM",
        "ERP",
        "Email Client",
        "IDE",
        "Browser"
    ]
}

In [103]:
EVENT_RESOURCE_MAPPING = {

    "Email": [
        "Corporate Mailbox"
    ],

    "File Access": [
        "Shared Drive",
        "Department Documents",
        "Cloud Storage"
    ],

    "Database Query": [
        "Employee Database",
        "Financial Database",
        "Customer Database"
    ],

    "Download": [
        "Project Files",
        "Reports",
        "Documents"
    ],

    "Upload": [
        "Cloud Storage",
        "SharePoint"
    ],

    "VPN Login": [
        "VPN Gateway"
    ],

    "Admin Action": [
        "Domain Controller",
        "Active Directory"
    ],

    "PowerShell Execution": [
        "Windows Server"
    ],

    "Print": [
        "Network Printer"
    ],

    "Login": [
        "Authentication Server"
    ],

    "Login Failed": [
        "Authentication Server"
    ],

    "Logout": [
        "Authentication Server"
    ],

    "Application Launch": [
    "VS Code",
    "Docker",
    "Confluence",
    "Browser",
    "Outlook"
]
}

# Behavioral Drift Simulation

Real employees do not follow identical routines every day.

This section introduces natural behavioral variations without labeling them as malicious.

Examples include:

- Slightly early or late logins
- Occasional remote work
- Device changes
- VPN usage
- Overtime
- Minor changes in application usage

These variations help the anomaly detection model distinguish between normal behavioral changes and genuine cyber threats.

In [47]:
def apply_behavioral_drift(events_df):

    df = events_df.copy()

    session_ids = df["session_id"].drop_duplicates()

    selected_sessions = session_ids.sample(
        frac=0.10,
        random_state=42
    )

    for session in selected_sessions:

        session_mask = df["session_id"] == session

        drift_type = random.choice([
            "late_login",
            "early_login",
            "remote_work",
            "device_change",
            "overtime"
        ])

        if drift_type == "late_login":

            df.loc[session_mask, "timestamp"] += timedelta(
                minutes=random.randint(15, 90)
            )

        elif drift_type == "early_login":

            df.loc[session_mask, "timestamp"] -= timedelta(
                minutes=random.randint(15, 60)
            )

        elif drift_type == "remote_work":

            df.loc[session_mask, "network_type"] = "Home WiFi"

        elif drift_type == "device_change":

            df.loc[session_mask, "device"] = random.choice([
                "Windows Laptop",
                "MacBook",
                "Linux Workstation"
            ])

        elif drift_type == "overtime":

            df.loc[session_mask, "timestamp"] += timedelta(
                hours=random.randint(1, 3)
            )

    return df

In [48]:
events_df = apply_behavioral_drift(events_df)

print("Behavioral drift successfully applied.")

Behavioral drift successfully applied.


# Cyber Attack Injection

This section transforms a subset of normal employee sessions into realistic cyber attack scenarios.

Instead of generating isolated malicious events, existing sessions are modified to simulate attacks such as:

- Credential Theft
- Brute Force
- Privilege Escalation
- Insider Threat
- Data Exfiltration
- Impossible Travel
- Malware Execution
- Lateral Movement

Each injected attack preserves realistic event sequences while introducing abnormal behavior for AI model training.

In [85]:
def select_attack_employees(events_df):

    employee_ids = events_df["employee_id"].unique()

    num_attack = int(
        len(employee_ids) * CONFIG["attack_percentage"]
    )

    return random.sample(list(employee_ids), num_attack)

In [86]:
attack_employees = select_attack_employees(events_df)

print(f"Employees Selected For Attack : {len(attack_employees)}")

Employees Selected For Attack : 50


# Attack Injection Functions

This section implements individual cyber attack scenarios.

Each attack modifies existing enterprise sessions to create realistic malicious behavior while preserving normal enterprise activity.

The attacks are injected into a subset of employees selected earlier.

Implemented attacks:

- Credential Theft
- Brute Force
- Privilege Escalation
- Insider Threat
- Data Exfiltration
- Impossible Travel
- Malware Execution
- Lateral Movement

In [123]:
def inject_credential_theft(df, employee_id):

    sessions = df.loc[
        df["employee_id"] == employee_id,
        "session_id"
    ].unique()

    if len(sessions) == 0:
        return df, None

    session_id = random.choice(sessions)

    session = df["session_id"] == session_id
    login = session & (df["event_type"] == "Login")

    df.loc[login, "timestamp"] -= pd.to_timedelta(
        random.randint(4, 7),
        unit="h"
    )

    df.loc[session, "network_type"] = "VPN"

    device = random.choice([
        "MacBook",
        "Linux Workstation",
        "Windows Laptop"
    ])

    df.loc[session, "device"] = device
    df.loc[session, "authentication"] = "Password"

    vpn_events = df[
        session &
        (df["event_type"] == "VPN Login")
    ].index.tolist()

    for idx in vpn_events:

        event = random.choice([
            "File Access",
            "Database Query",
            "Application Launch",
            "Download",
            "Email"
        ])

        df.at[idx, "event_type"] = event

        df.at[idx, "application"] = random.choice(
            EVENT_APPLICATION_MAPPING[event]
        )

        df.at[idx, "resource"] = random.choice(
            EVENT_RESOURCE_MAPPING[event]
        )

    rows = df[session].sort_values(
        "timestamp"
    ).index.tolist()

    second_half = rows[len(rows) // 2:]

    candidates = [
        idx for idx in second_half
        if df.at[idx, "event_type"] in [
            "Application Launch",
            "Email"
        ]
    ]

    if candidates:

        for idx in random.sample(
            candidates,
            min(3, len(candidates))
        ):

            event = random.choice([
                "Database Query",
                "File Access",
                "Download"
            ])

            df.at[idx, "event_type"] = event

            df.at[idx, "application"] = random.choice(
                EVENT_APPLICATION_MAPPING[event]
            )

            df.at[idx, "resource"] = random.choice([
                "Financial Database",
                "Customer Database",
                "Employee Database",
                "Shared Drive",
                "Cloud Storage"
            ])

    df.loc[session, "risk_baseline"] = "High"
    df.loc[session, "label"] = "Threat"
    df.loc[session, "attack_type"] = "Credential Theft"
    df.loc[session, "attack_severity"] = "High"

    df.loc[session, "attack_label"] = "Credential Theft"
    df.loc[session, "is_attack"] = 1

    return df, session_id

In [88]:
events_df["attack_type"] = None
events_df["attack_severity"] = None

In [89]:
test_employee = attack_employees[0]

events_df, attacked_session = inject_credential_theft(
    events_df,
    test_employee
)

print(f"Credential Theft injected for {test_employee}")
print(f"Attacked Session: {attacked_session}")

Credential Theft injected for EMP0145
Attacked Session: 5477893d-8810-4030-b35b-84066887849d


In [90]:
events_df[
    events_df["session_id"] == attacked_session
][[
    "session_id",
    "timestamp",
    "event_type",
    "network_type",
    "device",
    "authentication",
    "label",
    "attack_type",
    "attack_severity"
]].sort_values("timestamp")

,session_id,timestamp,event_type,network_type,device,authentication,label,attack_type,attack_severity
87238,5477893d-8810-4030-b35b-84066887849d,2026-07-07 04:53:38,Login,VPN,MacBook,Password,Threat,Credential Theft,High
87572,5477893d-8810-4030-b35b-84066887849d,2026-07-07 09:22:34,Admin Action,VPN,MacBook,Password,Threat,Credential Theft,High
87970,5477893d-8810-4030-b35b-84066887849d,2026-07-07 09:50:52,Admin Action,VPN,MacBook,Password,Threat,Credential Theft,High
88354,5477893d-8810-4030-b35b-84066887849d,2026-07-07 10:15:36,File Access,VPN,MacBook,Password,Threat,Credential Theft,High
88795,5477893d-8810-4030-b35b-84066887849d,2026-07-07 10:41:57,Admin Action,VPN,MacBook,Password,Threat,Credential Theft,High
89114,5477893d-8810-4030-b35b-84066887849d,2026-07-07 11:02:12,PowerShell Execution,VPN,MacBook,Password,Threat,Credential Theft,High
89653,5477893d-8810-4030-b35b-84066887849d,2026-07-07 11:34:08,PowerShell Execution,VPN,MacBook,Password,Threat,Credential Theft,High
90202,5477893d-8810-4030-b35b-84066887849d,2026-07-07 12:08:00,Application Launch,VPN,MacBook,Password,Threat,Credential Theft,High
90667,5477893d-8810-4030-b35b-84066887849d,2026-07-07 12:35:38,Admin Action,VPN,MacBook,Password,Threat,Credential Theft,High
91034,5477893d-8810-4030-b35b-84066887849d,2026-07-07 12:58:07,Admin Action,VPN,MacBook,Password,Threat,Credential Theft,High


## Brute Force Attack Injection

This section simulates a brute force attack by injecting multiple failed login attempts immediately before a successful login within a user's session. The attacked session is labeled as a threat while preserving the original user activity. These patterns will later be used for feature engineering and machine learning-based threat detection.

In [124]:
def inject_brute_force(df, employee_id):

    sessions = df.loc[
        df["employee_id"] == employee_id,
        "session_id"
    ].unique()

    if len(sessions) == 0:
        return df, None

    session_id = random.choice(sessions)

    session = df["session_id"] == session_id

    login_rows = df[
        session &
        (df["event_type"] == "Login")
    ]

    if login_rows.empty:
        return df, None

    login_row = login_rows.iloc[0]

    failed_attempts = random.randint(4, 8)

    failed_rows = []

    for i in range(failed_attempts):

        row = login_row.copy()

        row["event_id"] = str(uuid.uuid4())

        current_time = login_row["timestamp"]

        for _ in range(failed_attempts):

            current_time -= pd.Timedelta(
                seconds=random.randint(20, 60)
            )

            row = login_row.copy()

            row["event_id"] = str(uuid.uuid4())
            row["timestamp"] = current_time

        row["event_type"] = "Login Failed"

        row["application"] = random.choice(
            EVENT_APPLICATION_MAPPING["Login Failed"]
        )

        row["resource"] = random.choice(
            EVENT_RESOURCE_MAPPING["Login Failed"]
        )

        row["authentication"] = "Password"

        row["label"] = "Threat"
        row["attack_type"] = "Brute Force"
        row["attack_severity"] = "High"
        row["risk_baseline"] = "High"

        failed_rows.append(row)

    failed_df = pd.DataFrame(failed_rows)

    df = pd.concat(
        [df, failed_df],
        ignore_index=True
    )

    session = df["session_id"] == session_id

    df.loc[session, "authentication"] = "Password"
    df.loc[session, "risk_baseline"] = "High"
    df.loc[session, "label"] = "Threat"
    df.loc[session, "attack_type"] = "Brute Force"
    df.loc[session, "attack_severity"] = "High"

    df.loc[session, "attack_label"] = "Brute Force"
    df.loc[session, "is_attack"] = 1

    df = df.sort_values(
        "timestamp"
    ).reset_index(drop=True)

    return df, session_id

In [125]:
test_employee = attack_employees[1]

events_df, attacked_session = inject_brute_force(
    events_df,
    test_employee
)

In [126]:
events_df[
    events_df["session_id"] == attacked_session
][[
    "timestamp",
    "event_type",
    "application",
    "resource",
    "authentication",
    "label",
    "attack_type",
    "attack_severity"
]].sort_values("timestamp")

,timestamp,event_type,application,resource,authentication,label,attack_type,attack_severity
76340,2026-07-06 09:12:24,Login Failed,Authentication Service,Authentication Server,Password,Threat,Brute Force,High
76347,2026-07-06 09:12:49,Login Failed,Authentication Service,Authentication Server,Password,Threat,Brute Force,High
76350,2026-07-06 09:13:00,Login Failed,Authentication Service,Authentication Server,Password,Threat,Brute Force,High
76353,2026-07-06 09:13:05,Login Failed,Authentication Service,Authentication Server,Password,Threat,Brute Force,High
76389,2026-07-06 09:15:26,Login,Authentication Service,Authentication Server,Password,Threat,Brute Force,High
76950,2026-07-06 09:55:33,Upload,OneDrive,SharePoint,Password,Threat,Brute Force,High
77570,2026-07-06 10:33:24,Upload,OneDrive,Cloud Storage,Password,Threat,Brute Force,High
78203,2026-07-06 11:10:28,Download,SharePoint,Reports,Password,Threat,Brute Force,High
78757,2026-07-06 11:42:53,Download,Browser,Reports,Password,Threat,Brute Force,High
79252,2026-07-06 12:12:32,Download,Browser,Reports,Password,Threat,Brute Force,High


## Privilege Escalation Attack Injection

This section simulates a privilege escalation attack where a user gains elevated administrative privileges during an active session. The attack is characterized by privileged operations such as administrative actions, PowerShell execution, and access to sensitive resources. The compromised session is labeled as a threat while preserving realistic user activity.

In [127]:
def inject_privilege_escalation(df, employee_id):

    sessions = df.loc[
        df["employee_id"] == employee_id,
        "session_id"
    ].unique()

    if len(sessions) == 0:
        return df, None

    session_id = random.choice(sessions)

    session_mask = df["session_id"] == session_id

    session_df = (
        df.loc[session_mask]
        .sort_values("timestamp")
        .copy()
    )

    if len(session_df) < 5:
        return df, None

    middle_index = len(session_df) // 2
    middle_event = session_df.iloc[middle_index]
    base_time = middle_event["timestamp"]

    attack_events = []

    admin1 = middle_event.copy()

    admin1["event_id"] = str(uuid.uuid4())
    admin1["timestamp"] = base_time + pd.Timedelta(minutes=1)
    admin1["event_type"] = "Admin Action"
    admin1["application"] = random.choice(
        EVENT_APPLICATION_MAPPING["Admin Action"]
    )
    admin1["resource"] = random.choice(
        EVENT_RESOURCE_MAPPING["Admin Action"]
    )
    admin1["authentication"] = "Password"

    attack_events.append(admin1)

    powershell = middle_event.copy()

    powershell["event_id"] = str(uuid.uuid4())
    powershell["timestamp"] = base_time + pd.Timedelta(minutes=2)
    powershell["event_type"] = "PowerShell Execution"
    powershell["application"] = random.choice(
        EVENT_APPLICATION_MAPPING["PowerShell Execution"]
    )
    powershell["resource"] = random.choice(
        EVENT_RESOURCE_MAPPING["PowerShell Execution"]
    )
    powershell["authentication"] = "Password"

    attack_events.append(powershell)

    db_query = middle_event.copy()

    db_query["event_id"] = str(uuid.uuid4())
    db_query["timestamp"] = base_time + pd.Timedelta(minutes=3)
    db_query["event_type"] = "Database Query"
    db_query["application"] = random.choice(
        EVENT_APPLICATION_MAPPING["Database Query"]
    )
    db_query["resource"] = random.choice(
        EVENT_RESOURCE_MAPPING["Database Query"]
    )
    db_query["authentication"] = "Password"

    attack_events.append(db_query)

    admin2 = middle_event.copy()

    admin2["event_id"] = str(uuid.uuid4())
    admin2["timestamp"] = base_time + pd.Timedelta(minutes=4)
    admin2["event_type"] = "Admin Action"
    admin2["application"] = random.choice(
        EVENT_APPLICATION_MAPPING["Admin Action"]
    )
    admin2["resource"] = random.choice(
        EVENT_RESOURCE_MAPPING["Admin Action"]
    )
    admin2["authentication"] = "Password"

    attack_events.append(admin2)

    download = middle_event.copy()

    download["event_id"] = str(uuid.uuid4())
    download["timestamp"] = base_time + pd.Timedelta(minutes=5)
    download["event_type"] = "Download"
    download["application"] = random.choice(
        EVENT_APPLICATION_MAPPING["Download"]
    )
    download["resource"] = random.choice(
        EVENT_RESOURCE_MAPPING["Download"]
    )
    download["authentication"] = "Password"

    attack_events.append(download)

    attack_df = pd.DataFrame(attack_events)

    df = pd.concat([df, attack_df], ignore_index=True)

    df.loc[
        df["session_id"] == session_id,
        "attack_label"
    ] = "Privilege Escalation"

    df.loc[
        df["session_id"] == session_id,
        "is_attack"
    ] = 1

    df = (
        df.sort_values("timestamp")
          .reset_index(drop=True)
    )

    return df, session_id

## Data Exfiltration Attack Injection

This section simulates a data exfiltration attack where an insider accesses multiple sensitive resources and transfers large amounts of organizational data outside the network. The attack injects repeated file access, database queries, downloads, and uploads while preserving realistic user activity. The affected session is labeled as a threat for downstream machine learning tasks.

In [128]:
def inject_data_exfiltration(df, employee_id):

    sessions = df.loc[
        df["employee_id"] == employee_id,
        "session_id"
    ].unique()

    if len(sessions) == 0:
        return df, None

    session_id = random.choice(sessions)

    session_mask = df["session_id"] == session_id

    session_df = (
        df.loc[session_mask]
        .sort_values("timestamp")
        .copy()
    )

    if len(session_df) < 5:
        return df, None

    middle_index = len(session_df) // 2
    middle_event = session_df.iloc[middle_index]
    base_time = middle_event["timestamp"]

    attack_events = []

    db1 = middle_event.copy()

    db1["event_id"] = str(uuid.uuid4())
    db1["timestamp"] = base_time + pd.Timedelta(minutes=1)
    db1["event_type"] = "Database Query"
    db1["application"] = random.choice(
        EVENT_APPLICATION_MAPPING["Database Query"]
    )
    db1["resource"] = random.choice(
        EVENT_RESOURCE_MAPPING["Database Query"]
    )

    attack_events.append(db1)

    file_access = middle_event.copy()

    file_access["event_id"] = str(uuid.uuid4())
    file_access["timestamp"] = base_time + pd.Timedelta(minutes=2)
    file_access["event_type"] = "File Access"
    file_access["application"] = random.choice(
        EVENT_APPLICATION_MAPPING["File Access"]
    )
    file_access["resource"] = random.choice(
        EVENT_RESOURCE_MAPPING["File Access"]
    )

    attack_events.append(file_access)

    db2 = middle_event.copy()

    db2["event_id"] = str(uuid.uuid4())
    db2["timestamp"] = base_time + pd.Timedelta(minutes=3)
    db2["event_type"] = "Database Query"
    db2["application"] = random.choice(
        EVENT_APPLICATION_MAPPING["Database Query"]
    )
    db2["resource"] = random.choice(
        EVENT_RESOURCE_MAPPING["Database Query"]
    )

    attack_events.append(db2)

    download1 = middle_event.copy()

    download1["event_id"] = str(uuid.uuid4())
    download1["timestamp"] = base_time + pd.Timedelta(minutes=4)
    download1["event_type"] = "Download"
    download1["application"] = random.choice(
        EVENT_APPLICATION_MAPPING["Download"]
    )
    download1["resource"] = random.choice(
        EVENT_RESOURCE_MAPPING["Download"]
    )

    attack_events.append(download1)

    download2 = middle_event.copy()

    download2["event_id"] = str(uuid.uuid4())
    download2["timestamp"] = base_time + pd.Timedelta(minutes=5)
    download2["event_type"] = "Download"
    download2["application"] = random.choice(
        EVENT_APPLICATION_MAPPING["Download"]
    )
    download2["resource"] = random.choice(
        EVENT_RESOURCE_MAPPING["Download"]
    )

    attack_events.append(download2)

    upload = middle_event.copy()

    upload["event_id"] = str(uuid.uuid4())
    upload["timestamp"] = base_time + pd.Timedelta(minutes=6)
    upload["event_type"] = "Upload"
    upload["application"] = random.choice(
        EVENT_APPLICATION_MAPPING["Upload"]
    )
    upload["resource"] = random.choice(
        EVENT_RESOURCE_MAPPING["Upload"]
    )

    attack_events.append(upload)

    attack_df = pd.DataFrame(attack_events)

    df = pd.concat([df, attack_df], ignore_index=True)

    df.loc[
        df["session_id"] == session_id,
        "attack_label"
    ] = "Data Exfiltration"

    df.loc[
        df["session_id"] == session_id,
        "is_attack"
    ] = 1

    df = (
        df.sort_values("timestamp")
          .reset_index(drop=True)
    )

    return df, session_id

## Impossible Travel Attack Injection

This section simulates an impossible travel attack where the same user appears to log in from geographically distant locations within an unrealistically short time interval. The attack modifies login events to create an impossible travel pattern while preserving the rest of the session. The affected session is labeled as a threat for downstream machine learning tasks.

In [129]:
def inject_impossible_travel(df, employee_id):

    sessions = df.loc[
        df["employee_id"] == employee_id,
        "session_id"
    ].unique()

    if len(sessions) < 2:
        return df, None

    selected_sessions = random.sample(list(sessions), 2)

    session1 = selected_sessions[0]
    session2 = selected_sessions[1]

    login1 = df[
        (df["session_id"] == session1) &
        (df["event_type"] == "Login")
    ]

    login2 = df[
        (df["session_id"] == session2) &
        (df["event_type"] == "Login")
    ]

    if login1.empty or login2.empty:
        return df, None

    idx1 = login1.index[0]
    idx2 = login2.index[0]

    first_time = df.loc[idx1, "timestamp"]

    df.loc[idx2, "timestamp"] = (
        first_time + pd.Timedelta(minutes=random.randint(20, 45))
    )

    countries = [
        "India",
        "United States",
        "Germany",
        "Japan",
        "Australia",
        "United Kingdom",
        "Singapore",
        "Canada"
    ]

    first_country = random.choice(countries)

    second_country = random.choice(
        [c for c in countries if c != first_country]
    )

    df.loc[idx1, "country"] = first_country
    df.loc[idx2, "country"] = second_country

    df.loc[idx1, "city"] = "Unknown"
    df.loc[idx2, "city"] = "Unknown"

    df.loc[
        df["session_id"].isin([session1, session2]),
        "attack_label"
    ] = "Impossible Travel"

    df.loc[
        df["session_id"].isin([session1, session2]),
        "is_attack"
    ] = 1

    df = (
        df.sort_values("timestamp")
          .reset_index(drop=True)
    )

    return df, (session1, session2)

## Lateral Movement Attack Injection

This section simulates a lateral movement attack where an attacker compromises one workstation and then moves across multiple internal systems to expand access. The attack injects administrative operations, PowerShell execution, and repeated access to different resources while preserving realistic user behavior. The affected session is labeled as a threat for downstream machine learning tasks.

In [130]:
def inject_lateral_movement(df, employee_id):

    sessions = df.loc[
        df["employee_id"] == employee_id,
        "session_id"
    ].unique()

    if len(sessions) == 0:
        return df, None

    session_id = random.choice(sessions)

    session_mask = df["session_id"] == session_id

    session_df = (
        df.loc[session_mask]
        .sort_values("timestamp")
        .copy()
    )

    if len(session_df) < 5:
        return df, None

    middle_index = len(session_df) // 2
    middle_event = session_df.iloc[middle_index]
    base_time = middle_event["timestamp"]

    attack_events = []

    events = [
        "File Access",
        "Admin Action",
        "PowerShell Execution",
        "File Access",
        "Database Query"
    ]

    for i, event in enumerate(events):

        e = middle_event.copy()

        e["event_id"] = str(uuid.uuid4())
        e["timestamp"] = base_time + pd.Timedelta(minutes=i + 1)
        e["event_type"] = event
        e["application"] = random.choice(
            EVENT_APPLICATION_MAPPING[event]
        )
        e["resource"] = random.choice(
            EVENT_RESOURCE_MAPPING[event]
        )

        attack_events.append(e)

    attack_df = pd.DataFrame(attack_events)

    df = pd.concat([df, attack_df], ignore_index=True)

    df.loc[
        df["session_id"] == session_id,
        "attack_label"
    ] = "Lateral Movement"

    df.loc[
        df["session_id"] == session_id,
        "is_attack"
    ] = 1

    df = (
        df.sort_values("timestamp")
          .reset_index(drop=True)
    )

    return df, session_id

## Malicious Insider Attack Injection

This section simulates a malicious insider who intentionally accesses confidential information and transfers organizational data before ending the session. The attack injects repeated database queries, downloads, and uploads while preserving realistic user activity. The affected session is labeled as a threat for downstream machine learning tasks.

In [131]:
def inject_malicious_insider(df, employee_id):

    sessions = df.loc[
        df["employee_id"] == employee_id,
        "session_id"
    ].unique()

    if len(sessions) == 0:
        return df, None

    session_id = random.choice(sessions)

    session_mask = df["session_id"] == session_id

    session_df = (
        df.loc[session_mask]
        .sort_values("timestamp")
        .copy()
    )

    if len(session_df) < 5:
        return df, None

    middle_index = len(session_df) // 2
    middle_event = session_df.iloc[middle_index]
    base_time = middle_event["timestamp"]

    attack_events = []

    events = [
        "Database Query",
        "Database Query",
        "File Access",
        "Download",
        "Upload"
    ]

    for i, event in enumerate(events):

        e = middle_event.copy()

        e["event_id"] = str(uuid.uuid4())
        e["timestamp"] = base_time + pd.Timedelta(minutes=i + 1)
        e["event_type"] = event
        e["application"] = random.choice(
            EVENT_APPLICATION_MAPPING[event]
        )
        e["resource"] = random.choice(
            EVENT_RESOURCE_MAPPING[event]
        )

        attack_events.append(e)

    attack_df = pd.DataFrame(attack_events)

    df = pd.concat([df, attack_df], ignore_index=True)

    df.loc[
        df["session_id"] == session_id,
        "attack_label"
    ] = "Malicious Insider"

    df.loc[
        df["session_id"] == session_id,
        "is_attack"
    ] = 1

    df = (
        df.sort_values("timestamp")
          .reset_index(drop=True)
    )

    return df, session_id

In [139]:
ATTACK_FUNCTIONS = [
    inject_credential_theft,
    inject_brute_force,
    inject_privilege_escalation,
    inject_data_exfiltration,
    inject_impossible_travel,
    inject_lateral_movement,
    inject_malicious_insider
]

employee_ids = events_df["employee_id"].unique().tolist()

random.shuffle(employee_ids)

attack_summary = []

employees_per_attack = 10

current = 0

for attack_function in ATTACK_FUNCTIONS:

    selected = employee_ids[
        current: current + employees_per_attack
    ]

    current += employees_per_attack

    for employee_id in selected:

        events_df, attacked_session = attack_function(
            events_df,
            employee_id
        )

        if attacked_session is not None:

            attack_summary.append({
                "employee_id": employee_id,
                "attack_type": attack_function.__name__,
                "session": attacked_session
            })

In [153]:
events_df.to_csv(
    "synthetic_cybersecurity_logs.csv",
    index=False
)

print(f"Dataset Shape: {events_df.shape}")
print("Notebook 1 Completed Successfully!")

Dataset Shape: (239480, 21)
Notebook 1 Completed Successfully!
